In [1]:
## Import needed modules
import json
import gzip
import numpy as np
import pandas as pd
import plotly.express as px
import time

In [3]:
## Define Inputs
## The script below assumes you are reading in files from one specific date. The files will be named according to the convention above.
## Define starting and ending UTC times in the range of 000000Z to 235955Z. 

start = [0,0,0] # define UTC time for first file to be read [hh, mm, ss] i.e. the first file to be read and processed will be hhmmssZ.json.gz
end = [0,59,55] # define UTC time for last file to be read [hh, mm, ss]
inc = 5         # increment of time between snapshots/files.  This is normally 5 seconds, but hi-res data is avialable in 2 second intervals

# Change this to the path of your adsb files
path = "C:/Users/jacob/Documents/python_work/aia_work/test_data/"

# Data will be read into the following dictionary and then converted to a pandas dataframe
# The key values are a selection of what's available in the readsb-hist data files and the nameing conventions are those used by adsbexchange 
# See API here: for a complete list of potential key values.  For now I've selected a few that seem relevant
# The item values for each key will be lists that store the relevant data
processed_data = { 'flight': [],       #Flight name/number
                   'hex': [],          #ICAO identifier of the aircraft
                   't': [],            #Type of aircraft
                   'alt_baro': [],     #Barometric altitude
                   'baro_rate': [],    #Barometric climb/descent rate
                   'lat': [],          #Latitude
                   'lon': [],          #Longitude
                   'track': [],        #Ground track
                   'track_rate': [],   #Ground track rate of change
                   'gs': [],           #Ground speed
                   'mag_heading': [],  #Magnetic heading
                   'wd': [],           #Wind direction
                   'ws': [],           #Wind speed
                   'oat': [],          #Outside air temperature
                   'time': [],         #UTC time
                   }

In [5]:
## Function definitions

## Read the json.gz files and return the data as a python dictionary
def read_data(filename):
    with gzip.open(filename, 'rt', encoding='utf-8') as f:
        data = json.load(f) #read json data into a dictionary
    return data

## Take the data dictionary and pull values into the processed_data list defined above
def process_data(data):
    now = data['now'] #get the time stamp for this set of data in unix epoch time
    ac_data = data['aircraft'] #get a list of dictionaries that contain flight data for all airborne aircraft at the time stamp.  
                               #each dictionary in the list contains the data for one aircraft

    ## Convert the time to a readable formatted time string
    formatted_utc_time = time.strftime('%H:%M:%S', time.gmtime(now))

    for ac_dict in ac_data:
        ## Make sure the flight has an id and aircraft type
        if 'hex' not in ac_dict.keys() or 't' not in ac_dict.keys(): 
            continue # if not, just skip this aircraft for now

        # Pull values from data into dictionary that will be used to create dataframe
        for key in processed_data.keys():
            if key in ac_dict:                                  # Make sure this aircraft has the data we are trying to pull
                processed_data[key].append(ac_dict[key])
            elif key == 'time':                                 # Time is stored separaely from the aircraft dictionary
                processed_data[key].append(formatted_utc_time)
            else:
                processed_data[key].append(None)                # If the value doesn't exist, append None to keep all lists equal length

## To do: write some functions to claculate track, track rate, climb/descent, and maybe yaw

In [7]:
## Process data by looping through re-adsb data files
# This loops through all the files defined by the start and end zulu times and reads the data into processed_data dictionary that we defined
# earlier in the 'inputs' cell

for hh in range(start[0], end[0]+1):
    for mm in range(start[1], end[1]+1):
        for ss in range(start[2], end[2]+1, 5):
            zulu_time = str(hh).zfill(2)+str(mm).zfill(2)+str(ss).zfill(2)+'Z'  #Get the zulu time as a 6 digit string
            file_name = zulu_time+'.json.gz'                                    #append .json.gz to get the filename
            file_path = path+file_name                                          #append filename to folder path to get the complete path to the file
            data = read_data(file_path)                                         #read the json data into a python dictionary
            process_data(data)                                                  #process the data - pull values into the processed_data dict

In [ ]:
## Convert data to pandas dataframe and sort by aircraft hex id and time to group data by individual aircraft with data points organized by time 
df = pd.DataFrame(processed_data)
sorted_df = df.sort_values(['hex','time'], ascending=[False,True])

In [ ]:
## Find the most common aircraft types for this data set
unique_ids_df = sorted_df.drop_duplicates(subset='hex')  # Drop duplicates of hex id to get a list where each aircraft is captured once
ac_type_counts = unique_ids_df['t'].value_counts()       # Now find the value counts of 't' which is the type of aircraft
top_types = ac_type_counts[ac_type_counts > 100]         # Limit the list to aircraft types correspond to more than some threshold number of flights
filtered_type_df = sorted_df[sorted_df['t'].isin(top_types.keys())]  #filter the full dataframe by the most common types found above

print(top_types)

In [ ]:
## Plot the flights for a single type on a world map

## Choose an aircraft type from the list printed in the previous cell above
ac_type = 'DH8D' 

## Create a line plot showing ground tracks
fig1 = px.line_geo(filtered_type_df[filtered_type_df['t'] == ac_type],
                                      lat="lat",
                                      lon="lon",
                                      color="hex",
                                      projection='natural earth',
                                      hover_name="hex",)
fig1.show()

In [ ]:
## This cell plots a chosen variable vs time for a specific aircraft typ. 
## This plot isn't useful at all right now.  It's too messy to really glean much from it.  Need to think of a better way to visualize the data

ac_type = 'DH8D' ## Choose an aircraft type from the list printeder earlier
y = 'alt_baro'  ## Choose a variable to plot against time.  The variable list is the keys in processed_data that can be seen back in the inputs cell

#Plot y vs time for the aircraft type selected
fig2 = px.line(filtered_type_df[filtered_type_df['t'] == ac_type],
                  x='time',
                  y=y,
                  title=f"{y} vs time for {ac_type}")
fig2.show()